# Práctica 1 - Procesamiento del corpus

El notebook sigue el orden de la rúbrica. Los archivos `.txt` del corpus son la fuente original y no se sobrescriben durante el preprocesamiento.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CORPUS_DIR = ROOT / "corpus"
METADATA_PATH = CORPUS_DIR / "metadata.csv"
metadata = pd.read_csv(METADATA_PATH)
metadata.head()

## 1. Creación y caracterización del corpus

El corpus definitivo contiene 5.300 documentos: 5.000 del dataset de Kaggle y 300 obtenidos mediante web scraping. La generación se realiza con `src/get_corpus_dataset.py` y `src/scrape_corpus.py`; cada script conserva los archivos de la otra fuente.

In [ ]:
metadata["label_name"] = metadata["label"].map({0: "Falso", 1: "Verdad"})
metadata.groupby(["source_name", "label_name"]).size().unstack(fill_value=0)

In [ ]:
metrics = (
    metadata.groupby("source_name")["n_words"]
    .agg(
        count="count",
        mean="mean",
        std="std",
        min="min",
        max="max",
        q25=lambda values: values.quantile(0.25),
        q50=lambda values: values.quantile(0.50),
        q75=lambda values: values.quantile(0.75),
        sum="sum",
    )
    .rename(columns={"count": "Count", "mean": "Mean", "std": "Std", "min": "Min", "max": "Max", "q25": "Q25", "q50": "Q50", "q75": "Q75", "sum": "Sum"})
    .round(2)
)
metrics

In [ ]:
assert len(metadata) == 5300
assert metadata["label"].value_counts().to_dict() == {0: 2650, 1: 2650}
assert metadata["source_name"].value_counts().to_dict() == {"fuente2_kaggle": 5000, "newtral": 150, "bbc_mundo": 150}
assert all((CORPUS_DIR / row.file).exists() for row in metadata.itertuples())
print("Corpus válido: 5.300 documentos.")

## 2. Preprocesamiento

Esta fase se ejecuta después de crear el corpus. Incluye minúsculas, eliminación de tildes, números, URLs, saltos de línea, HTML, puntuación y emoticones. El resultado se guarda en una columna nueva para conservar el texto crudo.

In [ ]:
from src.preprocess import clean_text, preprocess_dataframe

# Carga explícita del texto original desde los TXT.
metadata["text"] = metadata["file"].map(
    lambda relative_path: (CORPUS_DIR / relative_path).read_text(encoding="utf-8")
)
processed = preprocess_dataframe(metadata, text_column="text", output_column="clean_text")
processed[["doc_id", "label_name", "text", "clean_text"]].head(3)

In [ ]:
# Verificación de que el preprocesamiento no sobrescribió los archivos fuente.
assert metadata["text"].str.len().sum() > processed["clean_text"].str.len().sum()
assert processed["clean_text"].map(lambda value: value == value.lower()).all()
assert not processed["clean_text"].str.contains(r"https?://|www\.", regex=True).any()
print("Preprocesamiento aplicado sobre una copia en memoria.")

## 3. POS Tagging

En esta sección se usará `src/pos_ner.py` con un modelo de spaCy para comparar NOUN, VERB, ADJ, ADV y PRON entre noticias falsas y verdaderas.

## 4. NER

En esta sección se extraerán entidades con spaCy y se almacenarán en una tabla con documento, etiqueta, texto y tipo de entidad.

## 5. Extracción de características

En esta sección se usarán unigramas, bigramas, stopwords, stemming, términos de ocurrencia y TF-IDF mediante `src/features.py`.

## 6. Modelado de temas

En esta sección se ejecutarán LSA, LDA y las métricas de coherencia mediante `src/topics.py`.

## 7. Clasificación

En esta sección se compararán cuatro algoritmos y ocho configuraciones usando validación cruzada mediante `src/classify.py`.